# plan-a-closure Tier 0 on a Colab GPU (engineering shakedown)

This runs the same one-command Tier 0 pilot (`closure tier0`) on whatever GPU this Colab runtime has, so the pipeline
gets checked on a **real device** instead of the fake NVML used in CI.

**What it is.** A shakedown on a non-target device, inside a virtual machine, with NVML as the only power source.
Precisions the GPU lacks (TF32 before Ampere, fp8 before compute capability 8.9) and clock or power controls the VM
refuses are dropped and noted in the logs.

**What it is not.** It is not the registered trial: there is no independent power meter, so assumption A6 does not
hold, and the registered numbers come from Tier 1 on instrumented H100/H200 hardware (`docs/PREREGISTRATION.md`).

**History.** The first Colab run (8 November 2026, version 0.5.4) stopped at `hw-selftest`: the unit of work was sized for an H100 and took half a second on a T4, so every load level ran flat out. Version 0.5.5 sizes the unit to the GPU (`docs/DEVIATIONS.md`). Cell 2 prints the version; it should say 0.5.5 or later.

**Before you start.** Runtime > Change runtime type > a GPU (the free T4 works). The full run takes about two hours;
keep this tab open so Colab does not disconnect the session. Nothing leaves the VM until you download the bundle.

In [ ]:
# 1. Which GPU did Colab give us?
import shutil, subprocess
if not shutil.which("nvidia-smi"):
    raise SystemExit("No GPU attached: Runtime > Change runtime type > pick a GPU, then run this cell again.")
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total,power.draw,power.limit", "--format=csv"],
    capture_output=True, text=True).stdout)

In [ ]:
# 2. Install (Colab already has a CUDA build of torch, which pip keeps).
!git clone --depth 1 https://github.com/AnonRish/plan-a-closure /content/plan-a-closure || git -C /content/plan-a-closure pull --ff-only
%cd /content/plan-a-closure
!pip install -q -e ".[test,hw]"
!python -c "import closure; print('plan-a-closure', closure.__version__)"

In [ ]:
# 3. Preflight: NVML, the energy counter, torch and CUDA, which physical GPU. If it reports no energy counter,
#    this runtime cannot run Tier 0; switch to another GPU type.
!closure preflight --gpu 0

Optional: a two-minute rehearsal on fake hardware, to see what the real run will produce.

In [ ]:
!closure tier0 --mock --skip-tests --out runs/rehearsal

## The real run (about two hours)

One unattended command. It logs every step, restores the GPU's clocks and power limit on exit, and writes
`SUMMARY.md` and `tier0_bundle.tar.gz` into a new folder under `runs/` (each run gets its own). Colab runs these commands as root, so no `sudo` is
needed. If the session drops, start again from cell 2. The `unit of work:` line at the top of the `hw-selftest` log shows the size chosen for this GPU (on a T4, about 4096 x 4096 float16).

In [ ]:
import time
OUT = time.strftime("runs/tier0-%Y%m%d-%H%M%S")   # a fresh folder, so a re-run never mixes with an old one
!closure tier0 --confirm-hardware --gpu 0 --out {OUT}

In [ ]:
import glob, os
if "OUT" not in globals():   # a restarted session forgets OUT: use the newest run folder
    runs = glob.glob("/content/plan-a-closure/runs/tier0-*")
    assert runs, "no run folder found: Colab replaced the machine, so run the notebook again from cell 2"
    OUT = max(runs, key=os.path.getmtime)
print("run folder:", OUT)
# 4. Read the summary here.
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/SUMMARY.md", encoding="utf-8").read()))

In [ ]:
import glob, os
if "OUT" not in globals():   # a restarted session forgets OUT: use the newest run folder
    runs = glob.glob("/content/plan-a-closure/runs/tier0-*")
    assert runs, "no run folder found: Colab replaced the machine, so run the notebook again from cell 2"
    OUT = max(runs, key=os.path.getmtime)
print("run folder:", OUT)
# 5. Download the bundle. It leaves out the run's private key (signing_key.bin) on purpose: never share that file.
from google.colab import files
files.download(f"{OUT}/tier0_bundle.tar.gz")

## Optional, after Tier 0: a mock training run (about 3 minutes)

A Narrow Path proposes that regulators test whether a compute provider can stop a training run at its permitted
compute. This drill runs a small training job on this GPU while a separate process, which sees only the NVML energy
counter, stops it when energy divided by the energy floor `hw-probe` measured reaches the budget. It needs
`probe.json` from the run above (`docs/TREATY_COVERAGE.md`).

In [ ]:
import glob, os
if "OUT" not in globals():   # a restarted session forgets OUT: use the newest run folder
    runs = glob.glob("/content/plan-a-closure/runs/tier0-*")
    assert runs, "no run folder found: Colab replaced the machine, so run the notebook again from cell 2"
    OUT = max(runs, key=os.path.getmtime)
print("run folder:", OUT)
!closure treaty mock-run --device cuda:0 --probe {OUT}/probe.json --budget-flop 1e16 --dim 4096 --layers 8 --batch 2048 --out {OUT}/mock-run
from google.colab import files
files.download(f"{OUT}/mock-run/drill.json")

## After the run

Run the summary and download cells as soon as the run ends: after about 90 idle minutes free Colab drops the session, and later it replaces the machine and its files. If only the session was dropped, the cells above find the run folder again.

* Publish the result whatever it shows, as the pre-registration requires, labelled as a shakedown.
* Append it to `docs/DEVIATIONS.md` as a new section, like section 1 there: date, GPU model, "Google Colab VM", no independent meter,
  which variants were dropped, and the bundle's SHA-256 (printed at the end of the run).
* The next step is the registered run: Tier 0 and then Tier 1 on instrumented H100/H200 hardware
  (`docs/HARDWARE_CONFIG.md`).